# Assessment 2: Machine Learning and Real Time Streaming
Student ID: 33602190  
Unit: ITO5202  
Year: 2026

This assessment uses the IBM AML HI-Small dataset from Assessment 1. Part A compares classification models and saves the selected model. Part B uses that model to predict transactions sent through Kafka.

## 0. Environment and configuration
Start Kafka with `bash scripts/start_kafka.sh` in a separate terminal. Open the notebook with `bash scripts/start_notebook.sh`, select Python (assessment2-aml) and run the cells in order.

File paths start from the project folder. The split uses seed 42, and Spark uses UTC for timestamps. Spark downloads the Kafka connector on its first run.

In [1]:
from pathlib import Path
import os
import sys
import platform
from importlib.metadata import version

PROJECT_ROOT = Path.cwd().resolve()
assert (PROJECT_ROOT / "requirements.txt").exists(), "Use the repo root."
JAVA_HOME = PROJECT_ROOT / ".runtime/jdk-17.0.20.1+1/Contents/Home"
assert (JAVA_HOME / "bin/java").exists(), "Install Java; see README."
expected_environment = (PROJECT_ROOT / ".venv").resolve()
assert Path(sys.prefix).resolve() == expected_environment, "Use the A2 kernel."
os.environ["JAVA_HOME"] = str(JAVA_HOME)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["PYSPARK_SUBMIT_ARGS"] = "--driver-memory 4g pyspark-shell"

SEED = 42
DATA_DIR = PROJECT_ROOT / "data"
MODEL_PATH = PROJECT_ROOT / "models/a2_model"
STREAM_PATH = DATA_DIR / "stream_data.parquet"
DEVELOPMENT_PATH = DATA_DIR / "development_data.parquet"
KAFKA_BOOTSTRAP_SERVERS = "127.0.0.1:9092"
KAFKA_TOPIC = "aml-transactions"
RUN_KAFKA_SMOKE_TEST = True
SPARK_KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.7"

assert platform.python_version() == "3.11.17"
assert version("pyspark") == "3.5.7"
print("Python:", platform.python_version())
print("PySpark:", version("pyspark"))
print("Kafka Python client:", version("kafka-python3"))
print("Kernel: assessment2-aml")

Python: 3.11.17
PySpark: 3.5.7
Kafka Python client: 3.0.0
Kernel: assessment2-aml


### Start Spark
Spark runs with four local threads and 4 GB of driver memory. The Kafka connector matches Spark 3.5.7 and Scala 2.12. The Kafka server uses its own Scala 2.13 build.

In [2]:
from pyspark.sql import SparkSession, functions as F

spark = (SparkSession.builder
         .master("local[4]")
         .appName("Assessment2-AML")
         .config("spark.jars.packages", SPARK_KAFKA_PACKAGE)
         .config("spark.jars.ivy", str(PROJECT_ROOT / ".runtime/ivy"))
         .config("spark.sql.session.timeZone", "UTC")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
jvm = spark.sparkContext._jvm
java_version = jvm.java.lang.System.getProperty("java.version")
scala_version = jvm.scala.util.Properties.versionNumberString()
assert java_version.startswith("17.")
assert scala_version.startswith("2.12.")
assert spark.range(10).agg(F.sum("id")).first()[0] == 45
print(f"PASS: Spark {spark.version}, Java {java_version}, Scala {scala_version}")
print("PASS: local Spark computation (sum 0–9 = 45)")

PASS: Spark 3.5.7, Java 17.0.20.1, Scala 2.12.18
PASS: local Spark computation (sum 0–9 = 45)


### Check Kafka and Structured Streaming
This check sends a sample JSON array through Kafka and reads it with Python and Spark. It removes the temporary topic when finished. Kafka needs to be running at `127.0.0.1:9092`.

Set `RUN_KAFKA_SMOKE_TEST = False` to skip this check when working offline.

In [3]:
from scripts.check_kafka import check_kafka

if RUN_KAFKA_SMOKE_TEST:
    setup_result = check_kafka(spark, PROJECT_ROOT, KAFKA_BOOTSTRAP_SERVERS)
    print(setup_result)
else:
    print("SKIPPED: Kafka connectivity check")

PASS: Python producer and consumer exchanged one JSON array.


PASS: Spark Structured Streaming parsed the same record using the Kafka connector.
{'python_round_trip': 'PASS', 'spark_streaming': 'PASS', 'records': 1}


## Part A: Machine learning
### A1. Load and validate the dataset

The task is binary classification using `is_laundering` as the target. A value of 1 means simulated laundering, and 0 means no laundering. The aim is to identify transactions that may need review.

The data comes from the HI-Small subset of [IBM Transactions for Anti-Money Laundering (AML)](https://www.kaggle.com/datasets/ealtman2019/ibm-transactions-for-anti-money-laundering-aml), which was approved for Assessment 1. Each row in the transaction file represents a transaction. The account file provides bank and account details.

#### Check the input files
The checksums are compared with those recorded in Assessment 1. The CSV headers are also checked before the columns are renamed.

In [4]:
import csv
import hashlib
from html import escape
from IPython.display import HTML, display


def show_table(headers, rows):
    header = "".join(f"<th>{escape(str(x))}</th>" for x in headers)
    body = "".join(
        "<tr>" + "".join(f"<td>{escape(str(x))}</td>" for x in row) + "</tr>"
        for row in rows
    )
    display(HTML(f"<table><thead><tr>{header}</tr></thead>"
                 f"<tbody>{body}</tbody></table>"))


expected_hashes = {
    "HI-Small_Trans.csv":
        "b19d39f515523373f991b689c07e11e7b0b95c17a2c27a87d91584ae16c5b040",
    "HI-Small_accounts.csv":
        "786808526e33cfc441212dd6fccda7edfc24172149bed59c6ef59b186836b014",
}
expected_headers = {
    "HI-Small_Trans.csv": [
        "Timestamp", "From Bank", "Account", "To Bank", "Account",
        "Amount Received", "Receiving Currency", "Amount Paid",
        "Payment Currency", "Payment Format", "Is Laundering",
    ],
    "HI-Small_accounts.csv": [
        "Bank Name", "Bank ID", "Account Number", "Entity ID", "Entity Name",
    ],
}
file_checks = []
for filename, expected_hash in expected_hashes.items():
    path = DATA_DIR / filename
    assert path.is_file(), f"Missing {filename}; see data/README.md."
    with path.open("rb") as file:
        actual_hash = hashlib.file_digest(file, "sha256").hexdigest()
    assert actual_hash == expected_hash, f"Checksum mismatch: {filename}"
    with path.open(newline="", encoding="utf-8-sig") as file:
        header = next(csv.reader(file))
    assert header == expected_headers[filename], f"Header mismatch: {filename}"
    file_checks.append((filename, "Matches Assessment 1", "Matches"))
show_table(["File", "SHA-256", "Header"], file_checks)

File,SHA-256,Header
HI-Small_Trans.csv,Matches Assessment 1,Matches
HI-Small_accounts.csv,Matches Assessment 1,Matches


#### Load the data
Account numbers are kept as strings. The two columns named `Account` are renamed `sender_account` and `receiver_account`. Bank IDs are numeric in both files. Loading stops if a record cannot be parsed. No rows are removed.

The timestamp format is `yyyy/MM/dd HH:mm`. The source does not specify a timezone, so UTC is used consistently without changing the recorded clock times. Amounts use double precision and are interpreted in their own currencies.

In [5]:
from pyspark import StorageLevel
from pyspark.sql.types import (
    StructType, StructField, TimestampType, LongType,
    StringType, DoubleType, IntegerType,
)

transaction_schema = StructType([
    StructField("timestamp", TimestampType(), True),
    StructField("from_bank", LongType(), True),
    StructField("sender_account", StringType(), True),
    StructField("to_bank", LongType(), True),
    StructField("receiver_account", StringType(), True),
    StructField("amount_received", DoubleType(), True),
    StructField("receiving_currency", StringType(), True),
    StructField("amount_paid", DoubleType(), True),
    StructField("payment_currency", StringType(), True),
    StructField("payment_format", StringType(), True),
    StructField("is_laundering", IntegerType(), True),
])
account_schema = StructType([
    StructField("bank_name", StringType(), True),
    StructField("bank_id", LongType(), True),
    StructField("account_number", StringType(), True),
    StructField("entity_id", StringType(), True),
    StructField("entity_name", StringType(), True),
])

transactions = (
    spark.read.schema(transaction_schema)
    .option("header", True).option("enforceSchema", True)
    .option("timestampFormat", "yyyy/MM/dd HH:mm")
    .option("mode", "FAILFAST")
    .csv(str(DATA_DIR / "HI-Small_Trans.csv"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
accounts = (
    spark.read.schema(account_schema)
    .option("header", True).option("enforceSchema", True)
    .option("mode", "FAILFAST")
    .csv(str(DATA_DIR / "HI-Small_accounts.csv"))
    .persist(StorageLevel.MEMORY_AND_DISK)
)
transaction_count = transactions.count()
account_count = accounts.count()
assert transaction_count == 5_078_345
assert account_count == 518_581
show_table(["Dataset", "Rows", "Columns"], [
    ("Transactions", f"{transaction_count:,}", len(transactions.columns)),
    ("Accounts", f"{account_count:,}", len(accounts.columns)),
])
show_table(["Dataset", "Field", "Spark type"], [
    (name, field.name, field.dataType.simpleString())
    for name, dataframe in [("Transactions", transactions), ("Accounts", accounts)]
    for field in dataframe.schema.fields
])

Dataset,Rows,Columns
Transactions,"5,078,345",11
Accounts,"518,581",5


Dataset,Field,Spark type
Transactions,timestamp,timestamp
Transactions,from_bank,bigint
Transactions,sender_account,string
Transactions,to_bank,bigint
Transactions,receiver_account,string
Transactions,amount_received,double
Transactions,receiving_currency,string
Transactions,amount_paid,double
Transactions,payment_currency,string
Transactions,payment_format,string


#### Check missing and invalid values
Check each column for nulls and text columns for blanks. Amounts must be finite and at least zero. Labels must be 0 or 1. The timestamp check also identifies missing dates after parsing.

In [6]:
missing_results = []
for name, dataframe in [("Transactions", transactions), ("Accounts", accounts)]:
    expressions = []
    for field in dataframe.schema.fields:
        missing = F.col(field.name).isNull()
        if isinstance(field.dataType, StringType):
            missing = missing | (F.trim(F.col(field.name)) == "")
        expressions.append(
            F.sum(F.when(missing, 1).otherwise(0)).alias(field.name)
        )
    counts = dataframe.agg(*expressions).first().asDict()
    missing_results.extend((name, column, count) for column, count in counts.items())
show_table(["Dataset", "Field", "Null or blank rows"], missing_results)
assert all(count == 0 for _, _, count in missing_results)

invalid_checks = {
    "invalid_labels": (
        F.col("is_laundering").isNull() | ~F.col("is_laundering").isin(0, 1)
    ),
}
for column in ["amount_paid", "amount_received"]:
    invalid_checks[f"invalid_{column}"] = (
        F.col(column).isNull() | F.isnan(column)
        | (F.abs(F.col(column)) == float("inf")) | (F.col(column) < 0)
    )
invalid_results = transactions.agg(*[
    F.sum(F.when(condition, 1).otherwise(0)).alias(name)
    for name, condition in invalid_checks.items()
]).first().asDict()
show_table(["Check", "Invalid rows"], invalid_results.items())
assert all(count == 0 for count in invalid_results.values())

Dataset,Field,Null or blank rows
Transactions,timestamp,0
Transactions,from_bank,0
Transactions,sender_account,0
Transactions,to_bank,0
Transactions,receiver_account,0
Transactions,amount_received,0
Transactions,receiving_currency,0
Transactions,amount_paid,0
Transactions,payment_currency,0
Transactions,payment_format,0


Check,Invalid rows
invalid_labels,0
invalid_amount_paid,0
invalid_amount_received,0


#### Check dates and labels
Compare the date range and label counts with Assessment 1. These checks cover the original files. Model development uses the separate development data created in A2.

In [7]:
# Keep the displayed dates in UTC.
coverage_display = transactions.agg(
    F.date_format(F.min("timestamp"), "yyyy-MM-dd HH:mm").alias("first"),
    F.date_format(F.max("timestamp"), "yyyy-MM-dd HH:mm").alias("last"),
).first()
show_table(["First transaction", "Last transaction", "Processing timezone"], [
    (coverage_display["first"], coverage_display["last"],
     spark.conf.get("spark.sql.session.timeZone")),
])
label_counts = transactions.groupBy("is_laundering").count().orderBy(
    "is_laundering"
).collect()
show_table(["Label", "Transactions", "Percent"], [
    (row.is_laundering, f"{row['count']:,}",
     f"{100 * row['count'] / transaction_count:.4f}%")
    for row in label_counts
])
assert {row.is_laundering: row["count"] for row in label_counts} == {
    0: 5_073_168, 1: 5_177,
}

First transaction,Last transaction,Processing timezone
2022-09-01 00:00,2022-09-18 16:18,UTC


Label,Transactions,Percent
0,"5,073,168",99.8981%
1,"5,177",0.1019%


#### Check account matches
An account is identified by its bank ID and account number together. Check that this combination is unique, then count transactions whose sender or receiver has no matching account. This helps prevent joins from duplicating or dropping transactions.

In [8]:
duplicate_account_keys = (
    accounts.groupBy("bank_id", "account_number").count()
    .filter(F.col("count") > 1).count()
)
account_keys = accounts.select("bank_id", "account_number")
unmatched_senders = transactions.join(
    account_keys,
    (transactions.from_bank == account_keys.bank_id)
    & (transactions.sender_account == account_keys.account_number),
    "left_anti",
).count()
unmatched_receivers = transactions.join(
    account_keys,
    (transactions.to_bank == account_keys.bank_id)
    & (transactions.receiver_account == account_keys.account_number),
    "left_anti",
).count()
join_checks = [
    ("Duplicate account key groups", duplicate_account_keys),
    ("Transactions without sender account", unmatched_senders),
    ("Transactions without receiver account", unmatched_receivers),
]
show_table(["Check", "Count"], join_checks)
assert all(count == 0 for _, count in join_checks)

Check,Count
Duplicate account key groups,0
Transactions without sender account,0
Transactions without receiver account,0


#### Check duplicate transactions
The source has no transaction ID. Compare the ten fields other than the label to find repeated transactions and any conflicting labels. Exact duplicates have the same values in all eleven fields. Keep all rows during these checks.

In [9]:
transaction_content_columns = [
    column for column in transactions.columns if column != "is_laundering"
]
content_groups = transactions.groupBy(*transaction_content_columns).agg(
    F.count("*").alias("source_rows"),
    F.countDistinct("is_laundering").alias("label_variants"),
)
duplicate_summary = content_groups.agg(
    F.sum(F.when(F.col("source_rows") > 1, 1).otherwise(0))
     .alias("repeated_content_groups"),
    F.sum(F.when(F.col("source_rows") > 1, F.col("source_rows")).otherwise(0))
     .alias("rows_in_repeated_groups"),
    F.sum(F.col("source_rows") - F.col("label_variants"))
     .alias("exact_duplicate_excess_rows"),
    F.sum(F.when(F.col("label_variants") > 1, 1).otherwise(0))
     .alias("conflicting_label_groups"),
).first().asDict()
show_table(["Check", "Count"], [
    (name.replace("_", " ").capitalize(), f"{count:,}")
    for name, count in duplicate_summary.items()
])
print("Source rows retained:", f"{transaction_count:,}")

Check,Count
Repeated content groups,9
Rows in repeated groups,18
Exact duplicate excess rows,9
Conflicting label groups,0


Source rows retained: 5,078,345


#### Validation results
The files contain 5,078,345 transactions and 518,581 accounts, matching Assessment 1. There are no missing values or invalid labels. Both amount fields contain finite values of zero or more. The dates run from 1 September 2022 at 00:00 to 18 September 2022 at 16:18, using UTC for processing.

Every sender and receiver matches an account. The combination of bank ID and account number is unique. There are 5,177 transactions labelled as laundering, or 0.1019% of all transactions. This imbalance needs to be considered when evaluating the models.

Nine pairs have identical transaction details and labels. This gives 18 rows in duplicate groups and nine extra duplicate rows. There are no conflicting labels. Without transaction IDs, it is not possible to tell whether these are duplicate entries or separate events with identical details, so all rows are kept. A2 keeps each pair in the same partition. Different transactions involving the same account can still appear in different partitions.

### A2. Split the development and streaming data

The split targets 75% for development and 25% for streaming. This meets the assignment minimums of 70% and 20%. The streaming data is reserved for Part B before fitting any preprocessing or model.

The nine duplicate pairs need to stay together. A SHA-256 key groups transactions using the ten fields other than the label. A hash calculation includes seed 42 as a salt to assign each group to a partition. The resulting percentages are checked because they will not be exactly 75% and 25%.

Within development, seed 43 is used to target 70% training, 15% validation and 15% testing. Duplicate pairs stay together here as well. Training data is used to fit preprocessing and models. Validation data is used to choose the model and prediction threshold. Test data is reserved for the final evaluation. The split mixes transaction dates and can include the same account in different partitions.

#### Add source row IDs
`source_record_id` is the transaction's position in the original CSV, starting at 1 after the header. Identical rows receive different IDs. Each record occupies one line in this file. The counts and contents are checked against A1 after adding the IDs. Keeping the original file order preserves these IDs.

In [10]:
raw_lines = spark.sparkContext.textFile(
    str(DATA_DIR / "HI-Small_Trans.csv"), minPartitions=4
)
indexed_lines = raw_lines.zipWithIndex().filter(lambda item: item[1] > 0)
numbered_lines = spark.createDataFrame(
    indexed_lines.map(lambda item: (item[1], item[0])),
    "source_record_id long, raw_csv string",
)
identified_transactions = (
    numbered_lines.select(
        "source_record_id",
        F.from_csv(
            "raw_csv", transaction_schema.simpleString(),
            {"mode": "FAILFAST", "timestampFormat": "yyyy/MM/dd HH:mm"},
        ).alias("record"),
    )
    .select("source_record_id", "record.*")
    .withColumn(
        "transaction_group_id",
        F.sha2(F.to_json(F.struct(*transaction_content_columns)), 256),
    )
    .persist(StorageLevel.DISK_ONLY)
)
id_summary = identified_transactions.agg(
    F.count("*").alias("rows"),
    F.countDistinct("source_record_id").alias("unique_ids"),
    F.min("source_record_id").alias("first_id"),
    F.max("source_record_id").alias("last_id"),
).first()
assert id_summary.rows == id_summary.unique_ids == transaction_count
assert id_summary.first_id == 1 and id_summary.last_id == transaction_count
assert transactions.exceptAll(
    identified_transactions.select(*transactions.columns)
).limit(1).count() == 0
show_table(["Check", "Result"], [
    ("Source rows and unique IDs", f"{id_summary.rows:,}"),
    ("First / last source ID", f"1 / {id_summary.last_id:,}"),
    ("Original transaction content", "Exact match, including duplicates"),
])
transactions.unpersist()
accounts.unpersist()

Check,Result
Source rows and unique IDs,"5,078,345"
First / last source ID,"1 / 5,078,345"
Original transaction content,"Exact match, including duplicates"


DataFrame[bank_name: string, bank_id: bigint, account_number: string, entity_id: string, entity_name: string]

#### Assign the partitions
The assignment uses the transaction group key and a fixed salt. It does not use the label or row position. Hash values are mapped to buckets from 0 to 999,999. With seed 42, the first 750,000 buckets go to development. Seed 43 divides development into training, validation and test data.

The row ID, group ID and partition columns are used to check the split. They are excluded from model features.

In [11]:
MODEL_SPLIT_SEED = 43
HASH_BUCKETS = 1_000_000


def hash_bucket(seed):
    return F.pmod(
        F.xxhash64("transaction_group_id", F.lit(seed)),
        F.lit(HASH_BUCKETS),
    )


def assign_partitions(dataframe):
    outer_bucket = hash_bucket(SEED)
    inner_bucket = hash_bucket(MODEL_SPLIT_SEED)
    return (
        dataframe.withColumn(
            "data_partition",
            F.when(outer_bucket < 750_000, "development").otherwise("stream"),
        )
        .withColumn(
            "model_partition",
            F.when(F.col("data_partition") == "stream", "stream")
             .when(inner_bucket < 700_000, "train")
             .when(inner_bucket < 850_000, "validation")
             .otherwise("test"),
        )
    )


partitioned_transactions = assign_partitions(
    identified_transactions
).persist(StorageLevel.DISK_ONLY)
outer_counts = {
    row.data_partition: row["count"]
    for row in partitioned_transactions.groupBy("data_partition").count().collect()
}
assert sum(outer_counts.values()) == transaction_count
assert outer_counts["development"] / transaction_count >= 0.70
assert outer_counts["stream"] / transaction_count >= 0.20
show_table(["Partition", "Rows", "Percent of source"], [
    (name, f"{outer_counts[name]:,}",
     f"{100 * outer_counts[name] / transaction_count:.4f}%")
    for name in ["development", "stream"]
])

Partition,Rows,Percent of source
development,"3,808,651",74.9979%
stream,"1,269,694",25.0021%


#### Check the split
Check that every source row appears once and that each transaction group belongs to only one partition. Repeat the assignment after changing the number of Spark partitions to confirm that the result stays the same.

In [12]:
partition_id_count = partitioned_transactions.select(
    "source_record_id"
).distinct().count()
assert partition_id_count == transaction_count
cross_partition_groups = (
    partitioned_transactions.groupBy("transaction_group_id")
    .agg(F.countDistinct("model_partition").alias("partitions"))
    .filter(F.col("partitions") != 1).count()
)
assert cross_partition_groups == 0
repeated_assignments = assign_partitions(
    identified_transactions.repartition(3)
).select("source_record_id", "data_partition", "model_partition")
assignment_changes = repeated_assignments.exceptAll(
    partitioned_transactions.select(
        "source_record_id", "data_partition", "model_partition"
    )
).limit(1).count()
assert assignment_changes == 0
show_table(["Check", "Result"], [
    ("Complete source-row coverage", f"{partition_id_count:,} unique IDs"),
    ("Content groups crossing partitions", cross_partition_groups),
    ("Assignment changes after repartitioning", assignment_changes),
])

Check,Result
Complete source-row coverage,"5,078,345 unique IDs"
Content groups crossing partitions,0
Assignment changes after repartitioning,0


#### Check the development groups
Count the labels in training, validation and test data to confirm that both classes are present. Streaming labels are not used to select features, tune models or evaluate Part A.

In [13]:
development_source = partitioned_transactions.filter(
    F.col("data_partition") == "development"
)
stream_source = partitioned_transactions.filter(F.col("data_partition") == "stream")
model_counts = {
    row.model_partition: row.asDict()
    for row in development_source.groupBy("model_partition").agg(
        F.count("*").alias("rows"),
        F.sum(F.when(F.col("is_laundering") == 0, 1).otherwise(0)).alias("negative"),
        F.sum(F.when(F.col("is_laundering") == 1, 1).otherwise(0)).alias("positive"),
    ).collect()
}
assert set(model_counts) == {"train", "validation", "test"}
assert sum(row["rows"] for row in model_counts.values()) == outer_counts["development"]
assert all(row["negative"] > 0 and row["positive"] > 0 for row in model_counts.values())
show_table(["Partition", "Rows", "% of development", "Label 0", "Label 1"], [
    (name, f"{model_counts[name]['rows']:,}",
     f"{100 * model_counts[name]['rows'] / outer_counts['development']:.4f}%",
     f"{model_counts[name]['negative']:,}", f"{model_counts[name]['positive']:,}")
    for name in ["train", "validation", "test"]
])

Partition,Rows,% of development,Label 0,Label 1
train,"2,664,035",69.9469%,"2,661,302","2,733"
validation,"572,159",15.0226%,"571,583",576
test,"572,457",15.0304%,"571,869",588


#### Save and reload the data
Save development records to `data/development_data.parquet` and streaming records to `data/stream_data.parquet`. Development records include their training, validation or test assignment. Spark creates a folder of Parquet files for each dataset. Running this cell again replaces these generated files.

Reload both datasets and compare their column types, counts and complete rows with the data before saving. Record the checksums, split settings, paths and counts in `data/split_manifest.json`.

In [14]:
development_source.write.mode("overwrite").parquet(str(DEVELOPMENT_PATH))
stream_source.write.mode("overwrite").parquet(str(STREAM_PATH))

development_df = spark.read.parquet(str(DEVELOPMENT_PATH))
stream_df = spark.read.parquet(str(STREAM_PATH))
reload_checks = []
for name, saved, original in [
    ("development", development_df, development_source),
    ("stream", stream_df, stream_source),
]:
    assert saved.dtypes == original.dtypes
    saved_count = saved.count()
    assert saved_count == outer_counts[name]
    assert original.exceptAll(
        saved.select(*original.columns)
    ).limit(1).count() == 0
    reload_checks.append((name, f"{saved_count:,}", "Exact match", "Match"))
show_table(["Reloaded dataset", "Rows", "Rows and IDs", "Field types"], reload_checks)

train_df = development_df.filter(F.col("model_partition") == "train")
validation_df = development_df.filter(F.col("model_partition") == "validation")
test_df = development_df.filter(F.col("model_partition") == "test")

import json

split_manifest = {
    "source_sha256": expected_hashes,
    "source_record_id": "1-based CSV record position, header excluded",
    "group_columns": transaction_content_columns,
    "method": "SHA-256 content key; xxhash64 with seed argument; modulo buckets",
    "outer_seed": SEED,
    "model_seed": MODEL_SPLIT_SEED,
    "hash_buckets": HASH_BUCKETS,
    "outer_development_threshold": 750_000,
    "model_train_validation_thresholds": [700_000, 850_000],
    "outer_counts": outer_counts,
    "development_partitions": model_counts,
    "paths": {
        "development": str(DEVELOPMENT_PATH.relative_to(PROJECT_ROOT)),
        "stream": str(STREAM_PATH.relative_to(PROJECT_ROOT)),
    },
    "checks": {
        "source_rows": transaction_count,
        "unique_source_ids": partition_id_count,
        "cross_partition_groups": cross_partition_groups,
        "assignment_changes": assignment_changes,
        "parquet_exact_match": True,
    },
}
(DATA_DIR / "split_manifest.json").write_text(
    json.dumps(split_manifest, indent=2, sort_keys=True) + "\n"
)
partitioned_transactions.unpersist()
identified_transactions.unpersist()
print("Saved data/split_manifest.json")

Reloaded dataset,Rows,Rows and IDs,Field types
development,"3,808,651",Exact match,Match
stream,"1,269,694",Exact match,Match


Saved data/split_manifest.json


#### Split results
Development contains 3,808,651 rows, or 74.9979% of the source. Streaming contains 1,269,694 rows, or 25.0021%. All 5,078,345 source rows are preserved, and both shares meet the assignment minimums.

Within development, there are 2,664,035 training rows, 572,159 validation rows and 572,457 test rows. These contain 2,733, 576 and 588 laundering labels respectively. Both classes are present in each group.

No duplicate group crosses a partition boundary. Changing the Spark partitions did not change any assignments. The saved Parquet files match the original assigned rows and column types. Use `train_df` for fitting, `validation_df` for model selection and `test_df` for final batch evaluation. Keep `stream_df` for Part B.

### A3. Explore the training data

### A4. Engineer features and build the pipeline

### A5. Train and compare models

### A6. Save and reload the selected pipeline

## Part B: Real time streaming
### B1. Kafka producer and message schema

### B2. Read, parse and predict

### B3. Sliding windows, watermark and outputs

### B4. Run the complete stream

### Reflection

### Stop Spark
Run this cell when finished. Start Spark again before continuing with more processing.

In [15]:
for query in spark.streams.active:
    query.stop()
spark.stop()
print("Spark session stopped.")

Spark session stopped.
